# YOLOv3 完整项目流程总结

## 一、项目概述

从零实现 YOLOv3 目标检测，在 `little_data`（4 类：人、猫、狗、马，18 张图片）上完成训练和推理。

---

## 二、完整流程

### 第 1 步：理论准备

| 概念 | 说明 |
|------|------|
| **YOLOv3 三部分损失** | 坐标损失 (MSE) + 置信度损失 (BCE) + 分类损失 (BCE) |
| **正负样本不平衡** | 每个图片 10647 个 anchor box，仅 ~9 个正样本 → 需要 λ_coord 和 λ_noobj 平衡 |
| **Anchor 机制** | 9 个预定义 anchor，分 3 组对应 3 个尺度 (13×13, 26×26, 52×52) |
| **NMS 后处理** | 从大量重叠候选框中为每个目标保留置信度最高的一个 |

### 第 2 步：标签编码 (Label Encoding)

**输入**: `Parse_label.txt` → `cls cx cy w h`（像素坐标）  
**输出**: 3 个尺度的标签张量 `[13,13,3,9]`, `[26,26,3,9]`, `[52,52,3,9]`

```
每个目标 → 每个尺度 → 选最佳 anchor → 编码 [conf=1, offset_x, offset_y, offset_w, offset_h, cls_onehot]

offset_x/y = 目标中心在 grid cell 内的偏移 (0~1)
offset_w/h = log(目标像素宽 / anchor 像素宽)
cls_onehot  = [人, 猫, 狗, 马] 的 one-hot 编码
```

### 第 3 步：网络结构

```
输入 (3, 416, 416)
  └─ Darknet-53 骨干网络 (52 层卷积)
       ├─ Stage 3 out → 52×52×256   (大目标检测)
       ├─ Stage 4 out → 26×26×512   (中目标检测)
       └─ Stage 5 out → 13×13×1024  (小目标检测)
  └─ FPN 特征金字塔
       ├─ 13×13 → ConvSet → 预测头 → out_13
       ├─ 上采样 → 与 26×26 拼接 → ConvSet → 预测头 → out_26
       └─ 上采样 → 与 52×52 拼接 → ConvSet → 预测头 → out_52
  └─ 输出: 3 × [B, 3×(5+4), H, W]
```

### 第 4 步：训练

```
优化器: Adam (lr=0.001, weight_decay=0.0005)
调度器: MultiStepLR [60, 90] gamma=0.1
损失:   coord_MSE × 5 + obj_BCE(正) + 0.25 × obj_BCE(负) + cls_BCE(正)
增强:   随机水平翻转
预训练: Darknet COCO 权重 → 迁移除最后预测层外的全部参数
Epochs: 100
```

### 第 5 步：推理

```
模型前向 → 3 个尺度原始输出
  └─ decode_scale: sigmoid(tx)+gx → cx, anchor_w×exp(tw) → bw
  └─ decode_all: 合并 10647 个候选框
  └─ post_process (NMS): 过滤 conf<0.5, 抑制 IoU>0.4 的同类框
  └─ 坐标还原: 416×416 → 原图尺寸
  └─ 绘制: PIL 绘制矩形框 + 中文标签
```

---

## 三、踩坑记录 & 修复

### Bug 1：分类损失 BCE(0,0) ≠ 0

```python
# ❌ 错误: 掩码方式导致背景贡献 ~29513 无效损失
loss_cls = BCE(cls_pred * cls_mask, cls_true * cls_mask)

# ✅ 正确: boolean indexing 只取正样本
loss_cls = BCE(cls_pred[pos_mask], cls_true[pos_mask])
```

**现象**: cls_loss 从 29520 → 29516 几乎不降，总 loss 在 29614 下不去

### Bug 2：置信度损失梯度冲突

```python
# ❌ 错误: 正样本同时被推往 1 和 0
loss_obj = BCE(pred, mask) + 0.5 × BCE(pred, 0)

# ✅ 正确: 正负样本分开计算
loss_obj = BCE(pred[pos], 1) + 0.25 × BCE(pred[neg], 0)
```

**现象**: obj_sigmoid 最高仅 0.196，无法通过 conf_thresh=0.3

### Bug 3：正负样本极度不平衡

- ~9 个正样本 vs ~10638 个负样本 → 梯度被负样本淹没
- **解决**: 改用 Adam（自适应学习率），正样本额外加权 `pos_weight = n_neg / (n_pos × 10)`
- **结果**: obj_sigmoid 最高从 0.196 → 0.8+

### Bug 4：OpenCV 不支持中文字体

- **解决**: 全部用 PIL 绘制（框 + 文字），最后转 OpenCV BGR 保存

---

## 四、实验结果

| 对比项 | 从头训练 (SGD) | 从头训练 (Adam) | COCO 预训练迁移 |
|--------|---------------|----------------|----------------|
| 最优损失 | 29614 | 87.0 | **51.0** |
| 检测结果 | 0 框 | 2~5 框/图 | 2~4 框/图 |
| 误检率 | - | 较高 | **较低** |
| 训练时间 | ~4min | ~4min | ~5min (含权重加载) |

**结论**: 预训练迁移有效，但 18 张图是准确率的主要瓶颈

---

## 五、YOLOv3 核心公式速查

### 损失函数

$$L = \lambda_{\text{coord}} \sum \mathbb{1}^{\text{obj}} \text{MSE}(t, \hat{t}) + \sum \mathbb{1}^{\text{obj}} \text{BCE}(C, \hat{C}) + \lambda_{\text{noobj}} \sum \mathbb{1}^{\text{noobj}} \text{BCE}(C, \hat{C}) + \sum \mathbb{1}^{\text{obj}} \sum_c \text{BCE}(p_c, \hat{p}_c)$$

### 解码公式

$$b_x = \sigma(t_x) + g_x \quad b_y = \sigma(t_y) + g_y \quad b_w = a_w e^{t_w} \quad b_h = a_h e^{t_h}$$

### IoU

$$\text{IoU} = \frac{|A \cap B|}{|A \cup B|}$$

---

## 六、改进方向

1. **收集更多数据**（最有效）
2. **更强的数据增强**: Mosaic、MixUp、随机裁剪
3. **调整 anchor**: 用 k-means 在 little_data 上重新聚类
4. **升级 YOLOv5/v8**: 内置更先进的训练策略
5. **更大模型 / 更久训练**: 但小数据集容易过拟合